# AI Chatbot - Model Training

This notebook demonstrates the complete machine learning workflow used to train the AI Chatbot.

## Objectives
- Load the chatbot dataset
- Preprocess user queries
- Convert text into numerical features using TF-IDF
- Train a Logistic Regression classifier
- Save the trained model
- Test predictions

**Tech Stack**
- Python
- NLTK
- Scikit-learn
- Flask

In [5]:
import json
import pickle
import nltk
import pandas as pd

from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [7]:
import json

with open("/content/intents.json","r",encoding="utf-8") as file:
    intents=json.load(file)

print("Number of Intents:",len(intents["intents"]))

Number of Intents: 21


In [8]:
patterns=[]
tags=[]

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        patterns.append(pattern)
        tags.append(intent["tag"])

df=pd.DataFrame({
    "Pattern":patterns,
    "Tag":tags
})

df.head(10)

,Pattern,Tag
0,Hi,greeting
1,Hello,greeting
2,Hey,greeting
3,Good morning,greeting
4,Good evening,greeting
5,Greetings,greeting
6,Hey there,greeting
7,Hi bot,greeting
8,How are you,greeting
9,What's up,greeting


# Data Preprocessing

The following preprocessing steps are performed:

- Convert text to lowercase
- Tokenization
- Lemmatization
- Remove punctuation

In [11]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab') # Added this line as per error suggestion

lemmatizer=WordNetLemmatizer()

clean_patterns=[]

for sentence in patterns:

    words=nltk.word_tokenize(sentence.lower())

    words=[
        lemmatizer.lemmatize(word)
        for word in words
        if word.isalnum()
    ]

    clean_patterns.append(" ".join(words))

clean_patterns[:10]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['hi',
 'hello',
 'hey',
 'good morning',
 'good evening',
 'greeting',
 'hey there',
 'hi bot',
 'how are you',
 'what up']

# TF-IDF Vectorization

TF-IDF converts text into numerical vectors that can be used by machine learning algorithms.

In [12]:
vectorizer=TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2)
)

X=vectorizer.fit_transform(clean_patterns)

print("Feature Matrix Shape:",X.shape)

Feature Matrix Shape: (122, 153)


In [13]:
encoder=LabelEncoder()

y=encoder.fit_transform(tags)

print("Classes:")

encoder.classes_

Classes:


array(['about', 'ai', 'college', 'css', 'date', 'flask', 'git', 'goodbye',
       'greeting', 'help', 'html', 'javascript', 'joke',
       'machine_learning', 'motivation', 'project', 'python', 'thanks',
       'time', 'unknown', 'weather'], dtype='<U16')

# Train Logistic Regression Model

In [14]:
model=LogisticRegression(
    max_iter=2000,
    random_state=42
)

model.fit(X,y)

print("Model Trained Successfully!")

Model Trained Successfully!


In [15]:
import os

os.makedirs("../model",exist_ok=True)

pickle.dump(model,open("../model/chatbot_model.pkl","wb"))
pickle.dump(vectorizer,open("../model/vectorizer.pkl","wb"))
pickle.dump(encoder,open("../model/label_encoder.pkl","wb"))

print("Model Saved Successfully!")

Model Saved Successfully!


# Test Predictions

In [16]:
queries=[
    "Hi",
    "What is Python?",
    "Explain AI",
    "Tell me a joke",
    "Thanks"
]

for query in queries:

    q=query.lower()

    words=nltk.word_tokenize(q)

    words=[
        lemmatizer.lemmatize(word)
        for word in words
        if word.isalnum()
    ]

    q=" ".join(words)

    vector=vectorizer.transform([q])

    prediction=model.predict(vector)[0]

    tag=encoder.inverse_transform([prediction])[0]

    print(f"Query: {query}")
    print(f"Predicted Intent: {tag}")
    print("-"*50)

Query: Hi
Predicted Intent: greeting
--------------------------------------------------
Query: What is Python?
Predicted Intent: python
--------------------------------------------------
Query: Explain AI
Predicted Intent: ai
--------------------------------------------------
Query: Tell me a joke
Predicted Intent: joke
--------------------------------------------------
Query: Thanks
Predicted Intent: thanks
--------------------------------------------------


# Conclusion

In this notebook:

- Loaded the chatbot dataset
- Preprocessed user queries
- Applied TF-IDF Vectorization
- Trained a Logistic Regression model
- Saved the trained model
- Tested predictions

The trained model is used by the Flask web application to generate chatbot responses.